# Hewwo Welcome to Fake or Real Review Project

# Step 1: Look at the big picture

# Step 2: Get the Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
import joblib
import random

In [2]:
reviews_df = pd.read_csv('fake reviews dataset.csv')

In [3]:
reviews_df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [4]:
reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  str    
 1   rating    40432 non-null  float64
 2   label     40432 non-null  str    
 3   text_     40432 non-null  str    
dtypes: float64(1), str(3)
memory usage: 1.2 MB


In [5]:
reviews_df.value_counts('label')

label
CG    20216
OR    20216
Name: count, dtype: int64

In [6]:
reviews_df.describe(include="all")


,category,rating,label,text_
count,40432,40432.000000,40432,40432
unique,10,NaN,2,40412
top,Kindle_Store_5,NaN,CG,Easy to put together and looks nice and the fi...
freq,4730,NaN,20216,2
mean,NaN,4.256579,NaN,NaN
std,NaN,1.144354,NaN,NaN
min,NaN,1.000000,NaN,NaN
25%,NaN,4.000000,NaN,NaN
50%,NaN,5.000000,NaN,NaN
75%,NaN,5.000000,NaN,NaN


In [7]:
reviews_df.value_counts("rating")

rating
5.0    24559
4.0     7965
3.0     3786
1.0     2155
2.0     1967
Name: count, dtype: int64

# clean the data

In [8]:
# drop rows that have the text missing
reviews_df.dropna(subset=['text_'], inplace=True)

#convert float64 ratings to integers
reviews_df["rating"] = reviews_df["rating"].astype(int)

In [10]:
from sklearn.preprocessing import LabelEncoder

# reviews_df.dropna(how="any") use this if eliminate any row that contains an usable value. i.e. 
# This is commented out because we want to keep rows that have feature other than text missing. To handle these values we will 
# use imputer rather than just deleting entire row.

# convert the labels of Computer Generated and Original to 0 and 1.

label_encoder = LabelEncoder()
label_encoder.fit(["CG","OR"])
reviews_df['label'] = label_encoder.transform(reviews_df['label'])

reviews_df.head()



,category,rating,label,text_
0,Home_and_Kitchen_5,5,0,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5,0,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5,0,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1,0,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5,0,Very nice set. Good quality. We have had the s...


# Split data into training and test data

In [11]:
# creates a split object, that represents 1 split, with 20% of the data being used for testing and 80 for training.
# random_state could be any number, it just ensures that the split is reproducible, 
# meaning that if you run the code multiple times, you will get the same split each time.
reviews_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# reviews_split makes sure there is similar or same percentage of training and test sets. 
# i.e. training set wont have 90% CG labels while testing set only have 10% CG labels.  
# train_index and test_index are two NumPy arrays of row indices
# we get the data frame for both sets by passing it into the locate function of our original dataframe
for train_index, test_index in reviews_split.split(reviews_df, reviews_df["label"]):
  training_set = reviews_df.loc[train_index]
  testing_set = reviews_df.loc[test_index]

# x and y train are used for supervised learning. Labels shown to the model
x_train = training_set.drop(columns = ["label"])
y_train = training_set["label"]

# x and y test used for evaluated the trained model on supervised learning. Labels are not shown to the model.
x_test = testing_set.drop(columns = ['label'])
y_test = testing_set["label"]

display(x_train.info(),y_train.info())

<class 'pandas.DataFrame'>
Index: 32345 entries, 25582 to 33847
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   category  32345 non-null  str  
 1   rating    32345 non-null  int64
 2   text_     32345 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1010.8 KB
<class 'pandas.Series'>
Index: 32345 entries, 25582 to 33847
Series name: label
Non-Null Count  Dtype
--------------  -----
32345 non-null  int64
dtypes: int64(1)
memory usage: 505.4 KB


None

None

# Step 4: Prepare the Data for Machine Learning algorithms

## Data preprocessing pipelines (transformations)

In [ ]:
#Use one-hot encodign to conver the category strings in the category column to numerical values that the model can underestand
#Use scaling to scale rating numerical 
#Use TF-IDF vectorizer to convert the review text into numerical features that the model can understand
#convert ratings to integers.


# note: might use small LLM to embed values for the text feature instead of TF-IDF to see which one gives more accurate results.

In [18]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

rating_feature = ['rating']
category_feature = ['category']
text_feature = 'text_'

#imputer handles the transformation of features that are either null or nan.
rating_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())

])
# one-hot converts column category into multiple columns with names of all values in the data. e.g. is_car, is_phone, is_radio instead of Category. Value in said rows
# are either 0 or 1 representing yes or no.
# unknown = "ignore" means that it will set all unknown values of new instances to 0s across the board for all the columns. 
category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value = "missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_pipeline = Pipeline([
  # not needed since we used dropna for all of invalid text values earlier
  # ("imputer", SimpleImputer(strategy="constant", fill_value="missing"))

  #tf-idf converts a string feature to a row of numbers. Each between 0-1 and represent each word. 
  # i.e. Each new column name is each word, instead of a single column named text
  ('tf-idf', TfidfVectorizer(stop_words="english", max_features=5000))

])

preprocessing = ColumnTransformer([
    ("ratings", rating_pipeline, rating_feature),
    ("categories", category_pipeline, category_feature),
    ("text", text_pipeline, text_feature)
])

In [ ]:
# Manual implementaion of one-hot encoder. Column Transformer actually does this automatically. Can delete this block, but i kept it
# for reference

# one-hot encoding for category feature
from sklearn.preprocessing import OneHotEncoder
# set sparse to False to get a dense array(numpy array object) instead of a sparse matrix
# handle_uknown will ignore any categories in the test set that were not seen in the training set and will not raise an error when evaluating categories that have not been seen.
category_encoder = OneHotEncoder(sparse_output=False, handle_unknown = 'ignore') 

# creates sparse matrix of one-hot encoded values for the category column
category_1hot_columns = category_encoder.fit_transform(reviews_df[['category']]) 
category_1hot_columns


# creates a new dataframe with the one-hot encoded values and column names base on the original category names
category_1hot_df = pd.DataFrame(category_1hot_columns, columns=category_encoder.get_feature_names_out())

# add the new columns to the original dataframe
reviews_df = pd.concat([reviews_df, category_1hot_df], axis =1)
reviews_df.head()
# delete the original "category"  column since we now have the one-hot encoded columns
reviews_df.drop(columns = ['category'])




In [ ]:
reviews_df.head()

# Step 5: Select a Model and Train it

## Baseline Model
we use the dummy model, because it is the minimum level intelligence that we need to beat to be considered useful



In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate


baseline_pipeline = Pipeline([
  ('prep', preprocessing),
  ('dummymodel', DummyClassifier(strategy="most_frequent"))
])

scoring = cross_validate(
  baseline_pipeline,
  x_train, y_train, scoring= ["accuracy", "precision", "recall", "average_precision"], cv = 5
)

scoring


/Users/matthewholguin/Desktop/Spring2026classes/MachineLearningIntro/Labs/lab-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/matthewholguin/Desktop/Spring2026classes/MachineLearningIntro/Labs/lab-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/matthewholguin/Desktop/Spring2026classes/MachineLearningIntro/Labs/lab-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to 

{'fit_time': array([1.77937698, 1.52683806, 1.52475715, 1.52099729, 1.50100517]),
 'score_time': array([0.80490804, 0.74223304, 0.7388339 , 0.73499584, 0.72594094]),
 'test_accuracy': array([0.49992271, 0.49992271, 0.49992271, 0.49992271, 0.49992271]),
 'test_precision': array([0.        , 0.        , 0.        , 0.49992271, 0.49992271]),
 'test_recall': array([0., 0., 0., 1., 1.]),
 'test_average_precision': array([0.50007729, 0.50007729, 0.50007729, 0.49992271, 0.49992271])}

# Step 6: Fine-tune the Model

# Step 7: Present the Solution

# Step 8: Launch, Monitor, and Maintain the System

In [ ]:
#first Code block y